In [75]:
import os
import pandas as pd
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [49]:
file_path = 'HDFS_2k.log'
line_count = 0

with open(file_path, 'r', encoding='utf-8') as file:
    log_lines = file.readlines()
print(f'\n Summation readed lines: {len(log_lines)}')


 Summation readed lines: 2000


Parse toàn bộ log với drain3

In [50]:
config = TemplateMinerConfig()
config.drain_sim_th = 0.4        # similarity threshold: 0.0-1.0
                                  # thấp = ít template (gộp nhiều), cao = nhiều template (tách nhiều)
config.drain_depth = 4            # độ sâu parse tree: 3-6
                                  # sâu hơn = chính xác hơn nhưng chậm hơn

miner = TemplateMiner(config=config)
cluster_data = []

for line in log_lines:
    result = miner.add_log_message(line.strip())
    # print(f"Cluster ID: {result['cluster_id']}")
    # print(f"Template:   {result['template_mined']}")
    # print(f"Change:     {result['change_type']}")
    # print()

print(f"Đã tạo ra tổng cộng {len(miner.drain.clusters)} templates (clusters).")

# Xem tất cả templates
print("=== Các Templates ===")
for cluster in miner.drain.clusters:
    # cluster_data.append()
    print(f"  [{cluster.cluster_id}] (count={cluster.size}): {cluster.get_template()}")
    cluster_data.append({
        'cluster_id': cluster.cluster_id,
        'template': cluster.get_template(),
        'count': cluster.size
    })
print('Append clusster xong!')

Đã tạo ra tổng cộng 17 templates (clusters).
=== Các Templates ===
  [1] (count=311): <*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating
  [2] (count=314): <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
  [3] (count=292): <*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>
  [4] (count=292): <*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>
  [5] (count=115): <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
  [6] (count=20): <*> <*> 13 INFO dfs.DataBlockScanner: Verification succeeded for <*>
  [7] (count=263): <*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>
  [8] (count=80): <*> <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
  [9] (count=80): <*> <*> <*> WARN dfs.DataNode$DataXceiver: <*> exception while serving <*> to <*>
  [10] (count=224): <*> <*> <

In [51]:
df = pd.DataFrame(cluster_data)
df

,cluster_id,template,count
0,1,<*> <*> <*> INFO dfs.DataNode$PacketResponder:...,311
1,2,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...,314
2,3,<*> <*> <*> INFO dfs.DataNode$PacketResponder:...,292
3,4,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Rec...,292
4,5,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...,115
5,6,<*> <*> 13 INFO dfs.DataBlockScanner: Verifica...,20
6,7,<*> <*> <*> INFO dfs.FSDataset: Deleting block...,263
7,8,<*> <*> <*> INFO dfs.DataNode$DataXceiver: <*>...,80
8,9,<*> <*> <*> WARN dfs.DataNode$DataXceiver: <*>...,80
9,10,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...,224


In [52]:
sorted_value = df.sort_values(by='count', ascending=False)
top10 = sorted_value.head(10)
top10

,cluster_id,template,count
1,2,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...,314
0,1,<*> <*> <*> INFO dfs.DataNode$PacketResponder:...,311
2,3,<*> <*> <*> INFO dfs.DataNode$PacketResponder:...,292
3,4,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Rec...,292
6,7,<*> <*> <*> INFO dfs.FSDataset: Deleting block...,263
9,10,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...,224
4,5,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...,115
8,9,<*> <*> <*> WARN dfs.DataNode$DataXceiver: <*>...,80
7,8,<*> <*> <*> INFO dfs.DataNode$DataXceiver: <*>...,80
5,6,<*> <*> 13 INFO dfs.DataBlockScanner: Verifica...,20


In [ ]:
os.makedirs('results', exist_ok=True)
csv_path = 'results/top_templates.csv'
top10.to_csv(csv_path, index=False)

## Phase 2

In [55]:
data = pd.read_csv(r'D:\XBrain Internship\02 AIOps\Week01\day02\results\top_templates.csv')
data

,cluster_id,template,count
0,2,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...,314
1,1,<*> <*> <*> INFO dfs.DataNode$PacketResponder:...,311
2,3,<*> <*> <*> INFO dfs.DataNode$PacketResponder:...,292
3,4,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Rec...,292
4,7,<*> <*> <*> INFO dfs.FSDataset: Deleting block...,263
5,10,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...,224
6,5,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...,115
7,9,<*> <*> <*> WARN dfs.DataNode$DataXceiver: <*>...,80
8,8,<*> <*> <*> INFO dfs.DataNode$DataXceiver: <*>...,80
9,6,<*> <*> 13 INFO dfs.DataBlockScanner: Verifica...,20


In [ ]:
def create_template_timeseries(log_entries, window='5min'):
    """
    Biến parsed log thành time series per template.
    
    Args:
        log_entries: list of (timestamp, template_id) tuples
        window: aggregation window
    
    Returns:
        DataFrame: columns = template IDs, rows = time windows, values = count
    """
    df = pd.DataFrame(log_entries, columns=['timestamp', 'template_id'])
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Group by time window + template → count
    ts = df.groupby([pd.Grouper(key='timestamp', freq=window), 'template_id']).size()
    ts = ts.unstack(fill_value=0)  # pivot: rows=time, cols=template
    
    return ts

In [64]:
log_entries = []

for line in log_lines:
    parts = line.split()
    if len(parts) >= 2:
        raw_time_str = f'{parts[0]} {parts[1]}'

        try: 
            timestamp = pd.to_datetime(raw_time_str, format='%y%m%d %H%M%S')
        except:
            continue

        result = miner.add_log_message(line)
        template_id = f"T-{result['cluster_id']}"
        
        log_entries.append((timestamp, template_id))

In [65]:
ts_df = create_template_timeseries(log_entries, window='5min')
display(ts_df.head())

template_id,T-1,T-10,T-11,T-12,T-13,T-14,T-15,T-16,T-17,T-2,T-3,T-4,T-5,T-6,T-7,T-8,T-9
timestamp,,,,,,,,,,,,,,,,,
2008-11-09 20:35:00,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2008-11-09 20:40:00,2,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0
2008-11-09 20:45:00,1,0,0,0,0,0,0,0,0,1,2,3,0,0,0,0,0
2008-11-09 20:50:00,1,0,0,0,0,0,0,0,0,1,2,0,2,0,0,0,0
2008-11-09 20:55:00,0,0,0,0,0,0,0,0,0,3,2,1,1,1,0,0,0


In [68]:
anomalies = []
for template_id in ts_df.columns:
    counts = ts_df[template_id]
    
    mean_val = counts.mean()
    std_val = counts.std()
    
    if std_val == 0:
        continue

        spikes = counts[(counts - mean_val) / std_val > 3]

    spikes = counts[(counts - mean_val) / std_val > 3]
    for timestamp, count in spikes.items():
        z_score = (count - mean_val) / std_val
        anomalies.append({
            'Thời gian (Window)': timestamp,
            'Template ID': template_id,
            'Số lượng (Count)': count,
            'Ngưỡng bình thường (Mean)': round(mean_val, 1),
            'Mức độ lệch (Z-Score)': round(z_score, 1)
        })

df_anomalies = pd.DataFrame(anomalies)

if df_anomalies.empty:
    print("Ổn định.")
else:
    df_anomalies = df_anomalies.sort_values(by='Mức độ lệch (Z-Score)', ascending=False)
    print(df_anomalies)

    Thời gian (Window) Template ID  Số lượng (Count)  \
22 2008-11-11 09:15:00        T-17                 1   
21 2008-11-11 08:05:00        T-16                 1   
19 2008-11-11 06:50:00        T-14                 1   
20 2008-11-11 06:50:00        T-15                 1   
14 2008-11-10 21:15:00        T-11                 1   
..                 ...         ...               ...   
1  2008-11-10 23:05:00         T-1                 5   
4  2008-11-11 07:30:00         T-1                 5   
2  2008-11-11 02:55:00         T-1                 5   
3  2008-11-11 07:20:00         T-1                 5   
0  2008-11-10 01:40:00         T-1                 5   

    Ngưỡng bình thường (Mean)  Mức độ lệch (Z-Score)  
22                        0.0                   17.4  
21                        0.0                   17.4  
19                        0.0                   17.4  
20                        0.0                   17.4  
14                        0.0                   17.4

In [73]:
templates = [cluster.get_template() for cluster in miner.drain.clusters]
print(f"Đang phân tích ma trận cho {len(templates)} templates...\n")

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(templates)

sim_matrix = cosine_similarity(tfidf_matrix)

print("Các cặp template có độ tương đồng cao (Cùng cụm/Cluster):")
for i in range(len(templates)):
    for j in range(i + 1, len(templates)):
        score = sim_matrix[i][j]
        if score > 0.5:  # Ngưỡng gom cụm
            print(f"- Score {score:.2f}:")
            print(f"  + T{i+1}: {templates[i]}")
            print(f"  + T{j+1}: {templates[j]}\n")

Đang phân tích ma trận cho 17 templates...

Các cặp template có độ tương đồng cao (Cùng cụm/Cluster):
- Score 0.52:
  + T2: <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
  + T10: <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>

- Score 0.51:
  + T3: <*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>
  + T13: 081111 <*> <*> INFO dfs.DataNode$DataXceiver: Received block <*> src: <*> dest: <*> of size 67108864

- Score 0.56:
  + T4: <*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>
  + T13: 081111 <*> <*> INFO dfs.DataNode$DataXceiver: Received block <*> src: <*> dest: <*> of size 67108864

- Score 0.93:
  + T14: 081111 065254 19 INFO dfs.FSNamesystem: BLOCK* ask 10.250.17.177:50010 to delete blk_-8570780307468499817 blk_-9122557405432088649 blk_-4393063808227796056 blk_8767569714374844347 blk_7079754042611

In [77]:
temp_log = '[2023-05-15T10:21:45.123Z] [payment-service-7b4f8c-2yvnm] [tid:c2f2e1] ERROR com.company.payment.DatabaseConnector - Connection timeout to db-primary at 10.0.1.5:5432 after 36000ms. Retries remaining: 2. Context: orderId=ORD-3634721, amount=18360, currency=VND'

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(templates)
semantic_sim_matrix = cosine_similarity(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5414.24it/s]


In [79]:
result = miner.add_log_message(temp_log)

In [82]:
if result['change_type'] == 'cluster_created':
    print("Hệ thống phát hiện hành vi hoàn toàn mới!")
    print(f"Drain3 đã tạo Template mới: {result['template_mined']}")
    print(f"ID của Template: T-{result['cluster_id']}")
elif result['change_type'] == 'none':
    print("THẤT BẠI: Drain3 đã gộp nhầm dòng log này vào một template cũ.")
    print(f"Nó bị nhét vào Template: {result['template_mined']}")

Hệ thống phát hiện hành vi hoàn toàn mới!
Drain3 đã tạo Template mới: [2023-05-15T10:21:45.123Z] [payment-service-7b4f8c-2yvnm] [tid:c2f2e1] ERROR com.company.payment.DatabaseConnector - Connection timeout to db-primary at 10.0.1.5:5432 after 36000ms. Retries remaining: 2. Context: orderId=ORD-3634721, amount=18360, currency=VND
ID của Template: T-18
